In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from skimage import feature as skfeature, exposure as skexposure

DATA_DIR = '../sample_data'
IMAGE_PATHS = sorted([os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith('.jpg')])
# Helper: load BGR image
def load(path):
    return cv2.imread(path)
# Label from filename: P1_B_52.jpg -> 'B'
def label(path):
    return os.path.basename(path).split('_')[1]

print(f'{len(IMAGE_PATHS)} images found')
for p in IMAGE_PATHS:
    print(os.path.basename(p))

# Preprocessing

## Resize (aspect-ratio preserving + center pad)

In [ ]:
img = cv2.imread("../sample_data/P1_B_52.jpg")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
print(img.shape)
plt.imshow(img_rgb)
plt.show()

In [ ]:
h, w = img.shape[:2]
size = 224
new_H, new_W = size, size
ratio = min(size / h, size /w )
new_h, new_w = int(h*ratio), int(w*ratio)
resize = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

img_rgb = cv2.cvtColor(resize, cv2.COLOR_BGR2RGB)
print(img_rgb.shape)
plt.imshow(img_rgb)
plt.show()

In [ ]:
h, w = img.shape[:2]
size = 224
new_H, new_W = size, size
ratio = min(new_H / h, new_W / w)
new_h, new_w = int(h*ratio), int(w*ratio)
print(h,w,'to', new_h,new_w)
resize = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
diff_h = (size - new_h) //2
diff_w = (size - new_w) // 2
blank_img = np.zeros((size, size, 3), dtype=np.uint8)
print(resize.shape, new_h, new_w)
blank_img[diff_h:diff_h+new_h, diff_w:diff_w+new_w] = resize
img_rgb = cv2.cvtColor(blank_img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)
plt.show()

In [ ]:
def resize_padded(img, size=224):
    h, w = img.shape[:2]
    ratio = min(size / h, size / w)
    new_h, new_w = int(h*ratio), int(w*ratio)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    dh, dw = (size - new_h) // 2, (size - new_w) // 2
    canvas[dh:dh+new_h, dw:dw+new_w] = resized
    return canvas

fig, ax = plt.subplots(5, 3, figsize=(9, 15))
ax = ax.flatten()
for j, path in enumerate(IMAGE_PATHS):
    img = load(path)
    r = resize_padded(img)
    ax[j].imshow(cv2.cvtColor(r, cv2.COLOR_BGR2RGB))
    ax[j].set_title(label(path))
    ax[j].axis('off')
plt.suptitle('Resized + Padded (224x224)', fontsize=14)
plt.tight_layout()
plt.show()

## Normalization Experiments: CLAHE vs Histogram Equalization

Compare: raw resize | CLAHE clip=2.0 | CLAHE clip=4.0 | global histogram equalization

Goal: find which method corrects uneven lighting without washing out skin color.

In [ ]:
def apply_clahe(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge([l_eq, a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

def apply_histeq(img):
    yuv = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)
    yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
    return cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)

# Pick 3 representative images: dark, medium, bright
sample_paths = [IMAGE_PATHS[0], IMAGE_PATHS[5], IMAGE_PATHS[10]]
methods = {
    'Raw resize': lambda x: x,
    'CLAHE clip=2.0': lambda x: apply_clahe(x, clip=2.0),
    'CLAHE clip=4.0': lambda x: apply_clahe(x, clip=4.0),
    'Hist EQ': apply_histeq,
}

fig, axes = plt.subplots(len(sample_paths), len(methods), figsize=(16, 12))
for row, path in enumerate(sample_paths):
    img = resize_padded(load(path))
    for col, (name, fn) in enumerate(methods.items()):
        out = fn(img.copy())
        axes[row, col].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
        if row == 0:
            axes[row, col].set_title(name, fontsize=11)
        if col == 0:
            axes[row, col].set_ylabel(label(path), fontsize=11)
        axes[row, col].axis('off')
plt.suptitle('Normalization Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/norm_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/norm_comparison.png')

## Blur Experiments: Gaussian vs Median

Compare: no blur | Gaussian 3x3 | Gaussian 5x5 | Median 5x5

Goal: suppress pixel noise while keeping hand edge sharpness for HOG.

In [ ]:
blur_methods = {
    'No blur': lambda x: x,
    'Gaussian 3x3': lambda x: cv2.GaussianBlur(x, (3,3), 0),
    'Gaussian 5x5': lambda x: cv2.GaussianBlur(x, (5,5), 0),
    'Median 5x5': lambda x: cv2.medianBlur(x, 5),
}

fig, axes = plt.subplots(len(sample_paths), len(blur_methods), figsize=(16, 12))
for row, path in enumerate(sample_paths):
    # Apply CLAHE first, then blur
    img = apply_clahe(resize_padded(load(path)))
    for col, (name, fn) in enumerate(blur_methods.items()):
        out = fn(img.copy())
        axes[row, col].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
        if row == 0:
            axes[row, col].set_title(name, fontsize=11)
        if col == 0:
            axes[row, col].set_ylabel(label(path), fontsize=11)
        axes[row, col].axis('off')
plt.suptitle('Blur Comparison (after CLAHE)', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/blur_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/blur_comparison.png')

# Segmentation

## Segmentation Experiments

Try 4 methods on 5 representative images. Show raw mask + masked overlay side by side.

1. YCrCb skin thresholding (Kovac 2003 range) + morphological cleanup
2. HSV skin thresholding + morphological cleanup
3. Otsu thresholding on grayscale
4. GrabCut (center-rect initialized)

**Best method** = cleanest hand isolation across varied backgrounds and skin tones.

In [ ]:
# Shared preprocessing for all segmentation experiments
def preprocess(img):
    r = resize_padded(img)
    return cv2.GaussianBlur(apply_clahe(r), (5,5), 0)

kernel_ellipse = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

def keep_largest(mask):
    """Return mask with only the largest connected component filled."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.zeros_like(mask), None
    largest = max(contours, key=cv2.contourArea)
    clean = np.zeros_like(mask)
    cv2.drawContours(clean, [largest], -1, 255, cv2.FILLED)
    return clean, largest

def overlay(img, mask, alpha=0.5):
    """Green overlay on masked region."""
    out = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    out[mask > 0] = (out[mask > 0] * (1 - alpha) + np.array([0, 200, 0]) * alpha).astype(np.uint8)
    return out

# Pick 5 diverse images
seg_paths = IMAGE_PATHS[:5]

### Method 1: YCrCb Skin Thresholding (Kovac 2003)

In [ ]:
LOWER_SKIN_YCRCB = np.array([0, 133, 77], dtype=np.uint8)
UPPER_SKIN_YCRCB = np.array([255, 173, 127], dtype=np.uint8)

def seg_ycrcb(img):
    ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
    mask = cv2.inRange(ycrcb, LOWER_SKIN_YCRCB, UPPER_SKIN_YCRCB)
    mask = cv2.erode(mask, kernel_ellipse, iterations=2)
    mask = cv2.dilate(mask, kernel_ellipse, iterations=2)
    return keep_largest(mask)

fig, axes = plt.subplots(len(seg_paths), 3, figsize=(12, 20))
for row, path in enumerate(seg_paths):
    img_pre = preprocess(load(path))
    clean_mask, contour = seg_ycrcb(img_pre)
    axes[row, 0].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_ylabel(label(path), fontsize=11)
    axes[row, 1].imshow(clean_mask, cmap='gray')
    axes[row, 2].imshow(overlay(img_pre, clean_mask))
    for ax in axes[row]: ax.axis('off')
axes[0,0].set_title('Preprocessed'); axes[0,1].set_title('YCrCb Mask'); axes[0,2].set_title('Overlay')
plt.suptitle('Segmentation: YCrCb Skin Thresholding', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/seg_ycrcb.png', dpi=120, bbox_inches='tight')
plt.show()

### Method 2: HSV Skin Thresholding

In [ ]:
def seg_hsv(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    # Skin hue: ~0-20 (red/orange) and 160-180 (wraps around)
    mask1 = cv2.inRange(hsv, np.array([0, 15, 50]), np.array([20, 170, 255]))
    mask2 = cv2.inRange(hsv, np.array([160, 15, 50]), np.array([180, 170, 255]))
    mask = cv2.bitwise_or(mask1, mask2)
    mask = cv2.erode(mask, kernel_ellipse, iterations=2)
    mask = cv2.dilate(mask, kernel_ellipse, iterations=2)
    return keep_largest(mask)

fig, axes = plt.subplots(len(seg_paths), 3, figsize=(12, 20))
for row, path in enumerate(seg_paths):
    img_pre = preprocess(load(path))
    clean_mask, contour = seg_hsv(img_pre)
    axes[row, 0].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_ylabel(label(path), fontsize=11)
    axes[row, 1].imshow(clean_mask, cmap='gray')
    axes[row, 2].imshow(overlay(img_pre, clean_mask))
    for ax in axes[row]: ax.axis('off')
axes[0,0].set_title('Preprocessed'); axes[0,1].set_title('HSV Mask'); axes[0,2].set_title('Overlay')
plt.suptitle('Segmentation: HSV Skin Thresholding', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/seg_hsv.png', dpi=120, bbox_inches='tight')
plt.show()

### Method 3: Otsu Thresholding on Grayscale

In [ ]:
def seg_otsu(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    mask = cv2.erode(mask, kernel_ellipse, iterations=2)
    mask = cv2.dilate(mask, kernel_ellipse, iterations=2)
    return keep_largest(mask)

fig, axes = plt.subplots(len(seg_paths), 3, figsize=(12, 20))
for row, path in enumerate(seg_paths):
    img_pre = preprocess(load(path))
    clean_mask, contour = seg_otsu(img_pre)
    axes[row, 0].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_ylabel(label(path), fontsize=11)
    axes[row, 1].imshow(clean_mask, cmap='gray')
    axes[row, 2].imshow(overlay(img_pre, clean_mask))
    for ax in axes[row]: ax.axis('off')
axes[0,0].set_title('Preprocessed'); axes[0,1].set_title('Otsu Mask'); axes[0,2].set_title('Overlay')
plt.suptitle('Segmentation: Otsu Thresholding', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/seg_otsu.png', dpi=120, bbox_inches='tight')
plt.show()

### Method 4: GrabCut (center-rect initialized)

In [ ]:
def seg_grabcut(img, margin=28):
    """GrabCut with rect covering the center 75% of the 224x224 image."""
    rect = (margin, margin, img.shape[1] - 2*margin, img.shape[0] - 2*margin)
    bgd_model = np.zeros((1, 65), np.float64)
    fgd_model = np.zeros((1, 65), np.float64)
    mask_gc = np.zeros(img.shape[:2], np.uint8)
    try:
        cv2.grabCut(img, mask_gc, rect, bgd_model, fgd_model, 5, cv2.GC_INIT_WITH_RECT)
        fg_mask = np.where((mask_gc == 2) | (mask_gc == 0), 0, 255).astype(np.uint8)
    except Exception:
        fg_mask = np.zeros(img.shape[:2], np.uint8)
    return keep_largest(fg_mask)

fig, axes = plt.subplots(len(seg_paths), 3, figsize=(12, 20))
for row, path in enumerate(seg_paths):
    img_pre = preprocess(load(path))
    clean_mask, contour = seg_grabcut(img_pre)
    axes[row, 0].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_ylabel(label(path), fontsize=11)
    axes[row, 1].imshow(clean_mask, cmap='gray')
    axes[row, 2].imshow(overlay(img_pre, clean_mask))
    for ax in axes[row]: ax.axis('off')
axes[0,0].set_title('Preprocessed'); axes[0,1].set_title('GrabCut Mask'); axes[0,2].set_title('Overlay')
plt.suptitle('Segmentation: GrabCut', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/seg_grabcut.png', dpi=120, bbox_inches='tight')
plt.show()

### Segmentation Side-by-Side Comparison

All 4 methods on the same 5 images for direct visual comparison.

In [ ]:
seg_methods = {
    'YCrCb': seg_ycrcb,
    'HSV': seg_hsv,
    'Otsu': seg_otsu,
    'GrabCut': seg_grabcut,
}

fig, axes = plt.subplots(len(seg_paths), len(seg_methods) + 1, figsize=(20, 20))
for row, path in enumerate(seg_paths):
    img_pre = preprocess(load(path))
    axes[row, 0].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_ylabel(label(path), fontsize=12)
    if row == 0: axes[row, 0].set_title('Original', fontsize=11)
    for col, (name, fn) in enumerate(seg_methods.items(), start=1):
        clean_mask, _ = fn(img_pre)
        axes[row, col].imshow(overlay(img_pre, clean_mask))
        if row == 0: axes[row, col].set_title(name, fontsize=11)
    for ax in axes[row]: ax.axis('off')
plt.suptitle('Segmentation Methods Comparison', fontsize=15)
plt.tight_layout()
plt.savefig('../sample_data/seg_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/seg_comparison.png')

### YCrCb Segmentation on All 15 Images

Uses the winning YCrCb method (adjust if HSV was better above).

In [ ]:
fig, axes = plt.subplots(5, 6, figsize=(18, 15))
axes = axes.reshape(5, 6)
for i, path in enumerate(IMAGE_PATHS):
    row, col = divmod(i, 3)
    img_pre = preprocess(load(path))
    clean_mask, _ = seg_ycrcb(img_pre)
    axes[row, col*2].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, col*2].set_title(f'{label(path)} (orig)', fontsize=9)
    axes[row, col*2].axis('off')
    axes[row, col*2+1].imshow(overlay(img_pre, clean_mask))
    axes[row, col*2+1].set_title(f'{label(path)} (seg)', fontsize=9)
    axes[row, col*2+1].axis('off')
plt.suptitle('YCrCb Segmentation — All 15 Images', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/seg_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/seg_results.png')

# Feature Extraction

## HOG Feature Experiments

Compare HOG at different `pixels_per_cell` values. Show gradient visualization.

Smaller cells = more local detail, larger vector. Larger cells = coarser, more compact.

In [ ]:
# Use 3 contrasting signs
hog_paths = [p for p in IMAGE_PATHS if label(p) in ('B', 'G', 'Y')][:3]
hog_configs = [
    {'pixels_per_cell': (8, 8),  'cells_per_block': (2, 2), 'label': 'ppc=8 (fine)'},
    {'pixels_per_cell': (16, 16), 'cells_per_block': (2, 2), 'label': 'ppc=16 (coarse)'},
]

fig, axes = plt.subplots(len(hog_paths), 1 + len(hog_configs), figsize=(14, 12))
for row, path in enumerate(hog_paths):
    img_pre = preprocess(load(path))
    gray = cv2.cvtColor(img_pre, cv2.COLOR_BGR2GRAY)
    axes[row, 0].imshow(gray, cmap='gray')
    axes[row, 0].set_ylabel(label(path), fontsize=12)
    if row == 0: axes[row, 0].set_title('Preprocessed (gray)')
    axes[row, 0].axis('off')
    for col, cfg in enumerate(hog_configs, start=1):
        feats, hog_img = skfeature.hog(
            gray,
            orientations=9,
            pixels_per_cell=cfg['pixels_per_cell'],
            cells_per_block=cfg['cells_per_block'],
            visualize=True,
            feature_vector=True
        )
        hog_vis = skexposure.rescale_intensity(hog_img, in_range=(0, 10))
        axes[row, col].imshow(hog_vis, cmap='gray')
        if row == 0: axes[row, col].set_title(f"HOG {cfg['label']}\nvec={feats.shape[0]}")
        axes[row, col].axis('off')
plt.suptitle('HOG Feature Visualization', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/hog_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/hog_comparison.png')

## LBP Feature Experiments

Local Binary Patterns — texture-based alternative to HOG.

Try radius=1 vs radius=3 to see texture granularity.

In [ ]:
from skimage.feature import local_binary_pattern

lbp_configs = [
    {'radius': 1, 'n_points': 8,  'label': 'r=1, p=8'},
    {'radius': 3, 'n_points': 24, 'label': 'r=3, p=24'},
]

fig, axes = plt.subplots(len(hog_paths), 1 + len(lbp_configs), figsize=(12, 12))
for row, path in enumerate(hog_paths):
    img_pre = preprocess(load(path))
    gray = cv2.cvtColor(img_pre, cv2.COLOR_BGR2GRAY)
    axes[row, 0].imshow(gray, cmap='gray')
    axes[row, 0].set_ylabel(label(path), fontsize=12)
    if row == 0: axes[row, 0].set_title('Preprocessed (gray)')
    axes[row, 0].axis('off')
    for col, cfg in enumerate(lbp_configs, start=1):
        lbp_img = local_binary_pattern(gray, cfg['n_points'], cfg['radius'], method='uniform')
        axes[row, col].imshow(lbp_img, cmap='gray')
        if row == 0: axes[row, col].set_title(f"LBP {cfg['label']}")
        axes[row, col].axis('off')
plt.suptitle('LBP Feature Visualization', fontsize=14)
plt.tight_layout()
plt.savefig('../sample_data/lbp_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/lbp_comparison.png')

## Contour Descriptors

From the segmented hand mask, extract shape descriptors:
- Bounding rect + convex hull drawn on image
- Convexity defects (finger gaps)
- Scalar features: area, perimeter, solidity, aspect ratio, extent, defect count

In [ ]:
def draw_contour_features(img, contour):
    """Draw hull, bounding rect, and convexity defects on a copy of img."""
    out = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    hull_pts = cv2.convexHull(contour)
    cv2.drawContours(out, [contour], -1, (0, 255, 0), 2)     # green: contour
    cv2.drawContours(out, [hull_pts], -1, (255, 165, 0), 2)  # orange: hull
    x, y, w, h = cv2.boundingRect(contour)
    cv2.rectangle(out, (x,y), (x+w, y+h), (255, 0, 0), 2)   # red: bounding rect
    # Convexity defects
    try:
        hull_idx = cv2.convexHull(contour, returnPoints=False)
        defects = cv2.convexityDefects(contour, hull_idx)
        if defects is not None:
            for d in defects:
                s, e, f, depth = d[0]
                if depth / 256.0 > 5:
                    far = tuple(contour[f][0])
                    cv2.circle(out, far, 5, (255, 0, 255), -1)  # magenta: defect tips
    except cv2.error:
        pass
    return out

def extract_contour_features(contour):
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity = area / hull_area if hull_area > 0 else 0
    x, y, w, h = cv2.boundingRect(contour)
    aspect_ratio = float(w) / h if h > 0 else 0
    extent = area / (w * h) if (w * h) > 0 else 0
    num_defects = 0
    try:
        hull_idx = cv2.convexHull(contour, returnPoints=False)
        defects = cv2.convexityDefects(contour, hull_idx)
        if defects is not None:
            num_defects = sum(1 for d in defects if d[0][3] / 256.0 > 5)
    except cv2.error:
        pass
    return {'area': area, 'perimeter': perimeter, 'solidity': round(solidity, 3),
            'aspect_ratio': round(aspect_ratio, 3), 'extent': round(extent, 3),
            'num_defects': num_defects}

# Visualize on 5 images
fig, axes = plt.subplots(len(seg_paths), 2, figsize=(10, 20))
for row, path in enumerate(seg_paths):
    img_pre = preprocess(load(path))
    clean_mask, contour = seg_ycrcb(img_pre)
    axes[row, 0].imshow(cv2.cvtColor(img_pre, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_ylabel(label(path), fontsize=12)
    if row == 0: axes[row, 0].set_title('Preprocessed')
    axes[row, 0].axis('off')
    if contour is not None:
        drawn = draw_contour_features(img_pre, contour)
        feats = extract_contour_features(contour)
        axes[row, 1].imshow(drawn)
        axes[row, 1].set_title(f"sol={feats['solidity']} ar={feats['aspect_ratio']} def={feats['num_defects']}",
                               fontsize=8)
    axes[row, 1].axis('off')
axes[0,1].set_title('Contour Features', fontsize=11)
plt.suptitle('Contour Descriptors (green=contour, orange=hull, red=bbox, magenta=defects)', fontsize=12)
plt.tight_layout()
plt.savefig('../sample_data/contour_features.png', dpi=120, bbox_inches='tight')
plt.show()

### Shape Features Table — All 15 Images

In [ ]:
import warnings
rows = []
for path in IMAGE_PATHS:
    img_pre = preprocess(load(path))
    _, contour = seg_ycrcb(img_pre)
    if contour is not None:
        f = extract_contour_features(contour)
        rows.append([label(path), f['area'], f['perimeter'], f['solidity'],
                     f['aspect_ratio'], f['extent'], f['num_defects']])
    else:
        rows.append([label(path), 0, 0, 0, 0, 0, 0])

col_labels = ['Sign', 'Area', 'Perimeter', 'Solidity', 'Aspect Ratio', 'Extent', 'Defects']
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')
tbl = ax.table(cellText=rows, colLabels=col_labels, loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.5)
plt.title('Shape Features — All 15 Sample Images', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig('../sample_data/shape_features.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/shape_features.png')

## Hu Moments

7 shape-invariant moments (log-transformed for numerical stability).

Compare across 3 sign classes to see how well they separate.

In [ ]:
def extract_hu(contour):
    M = cv2.moments(contour)
    hu = cv2.HuMoments(M).flatten()
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)

hu_paths = [p for p in IMAGE_PATHS if label(p) in ('B', 'G', 'Y', 'C', 'S')][:5]
hu_data = {}
for path in hu_paths:
    img_pre = preprocess(load(path))
    _, contour = seg_ycrcb(img_pre)
    if contour is not None:
        hu_data[label(path)] = extract_hu(contour)

x = np.arange(7)
width = 0.15
fig, ax = plt.subplots(figsize=(12, 5))
for i, (sign, hu) in enumerate(hu_data.items()):
    ax.bar(x + i * width, hu, width, label=f'Sign {sign}')
ax.set_xlabel('Hu Moment Index')
ax.set_ylabel('Log-transformed Value')
ax.set_title('Hu Moments per Sign Class')
ax.set_xticks(x + width * (len(hu_data)-1) / 2)
ax.set_xticklabels([f'h{i+1}' for i in range(7)])
ax.legend()
plt.tight_layout()
plt.savefig('../sample_data/hu_moments.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_data/hu_moments.png')

# Summary — Chosen Methods

After running the experiments above, record findings here before implementing `pipeline.py`.

| Stage | Chosen Method | Reason |
|-------|--------------|--------|
| Normalization | CLAHE clip=2.0, LAB L-channel | Tile-local correction without color distortion |
| Blur | Gaussian 5×5 | Suppresses noise, preserves hand edges |
| Segmentation | YCrCb skin thresholding | Most consistent across varied backgrounds/skin tones |
| Feature 1 | HOG ppc=(8,8) | Captures gradient orientation = hand pose |
| Feature 2 | Contour descriptors | Global shape (solidity, defects) |
| Feature 3 | Hu Moments | Scale/rotation invariance |